In [1]:
import pandas as pd

In [1]:
#linux command to download zipped data in runtime
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

In [2]:
#unzip zipped data
!unzip /content/new_articles.zip -d /content/drive/MyDrive/new_articles

Archive:  /content/new_articles.zip
replace /content/drive/MyDrive/new_articles/05-07-fintech-space-continues-to-be-competitive-and-drama-filled.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [4]:
import os

In [5]:
#merging content of different text files present in directory, to a single text file
source_dir = '/content/drive/MyDrive/new_articles'
output_dir = '/content/drive/MyDrive/merged_articles_content.txt'
files = [f for f in os.listdir(source_dir) if f.endswith('.txt')]
for file in files:
  file_path = os.path.join(source_dir, file)
  with open(file_path, 'r') as f:
    content = f.read()
  with open(output_dir, 'a') as f:
    f.write(content)

In [6]:
#converting raw text to small sized chunks
with open(output_dir, 'r') as f:
  content = f.read()
def chunk_text(text,chunk_size= 200):
  words = text.split()
  chunks = []
  for i in range(len(words)-chunk_size):
    chunk = ' '.join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks
chunks = chunk_text(content)

In [10]:
chunks[0:5]

['Welcome to The Interchange! If you received this in your inbox, thank you for signing up and your vote of confidence. If you’re reading this as a post on our site, sign up here so you can receive it directly in the future. Every week, we’ll take a look at the hottest fintech news of the previous week. This will include everything from funding rounds to trends to an analysis of a particular space to hot takes on a particular company or phenomenon. There’s a lot of fintech news out there and it’s our job to stay on top of it — and make sense of it — so you can stay in the know. — Mary Ann and Christine Busy, busy, busy It was a busy week in startup and venture lands, and the fintech space was no exception. In the venture world, I reported on Peter Ackerson’s departure from Fin Capital earlier this year and the fact that he has since started a new venture firm called Audere Capital. The circumstances around his departure remain fuzzy, but one source speculated that tension arose between

In [7]:
#using pre trained transformer for embedding chunks of data
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embedding = model.encode(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
embedding[0:5]

array([[-0.05279564, -0.12433624, -0.00276254, ..., -0.11289125,
        -0.01091943,  0.00681925],
       [-0.04978026, -0.12087627, -0.00184563, ..., -0.11149438,
         0.00106298,  0.0010008 ],
       [-0.05393552, -0.12867293, -0.00726829, ..., -0.10883208,
         0.00839088,  0.00630105],
       [-0.05186214, -0.12447664, -0.01020243, ..., -0.10159786,
         0.00803297,  0.00650516],
       [-0.04680544, -0.12214423, -0.00817147, ..., -0.10789344,
         0.01295997,  0.00766275]], dtype=float32)

In [13]:
embedding.shape

(27658, 384)

In [8]:
#mapping chunk data to index
doc_store = {
    i: chunks[i]
    for i in range(len(chunks))
}

In [15]:
doc_store[5]

'you received this in your inbox, thank you for signing up and your vote of confidence. If you’re reading this as a post on our site, sign up here so you can receive it directly in the future. Every week, we’ll take a look at the hottest fintech news of the previous week. This will include everything from funding rounds to trends to an analysis of a particular space to hot takes on a particular company or phenomenon. There’s a lot of fintech news out there and it’s our job to stay on top of it — and make sense of it — so you can stay in the know. — Mary Ann and Christine Busy, busy, busy It was a busy week in startup and venture lands, and the fintech space was no exception. In the venture world, I reported on Peter Ackerson’s departure from Fin Capital earlier this year and the fact that he has since started a new venture firm called Audere Capital. The circumstances around his departure remain fuzzy, but one source speculated that tension arose between Ackerson and Fin founding partn

In [9]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.1 MB/s eta 0:00:00


In [10]:
#implementing chunk indexing, using HNSW algorithm which helps in efficient fast search among chunks
import faiss
import numpy as np

embeddings = np.array(
    embedding
).astype('float32')

dimension = embeddings.shape[1]
index = faiss.IndexHNSWFlat(
    dimension, 32
)

index.add(embeddings)

In [18]:
#query embedding, user query is also embedded into numerical features
query_embedding = model.encode(
    ["What is Servicenow building with LLMs?"]
)

In [19]:
#computing distance between chunks and user query, top 3 chunk's index and distance are shown
distances, indices = index.search(
    np.array(query_embedding).astype('float32'),
    k=3
)

In [20]:
distances

array([[1.2860771, 1.3018172, 1.3032212]], dtype=float32)

In [21]:
indices

array([[17064, 17065, 17063]])

In [22]:
#retrieving top 3 similar data chunks
results = []

for idx in indices[0]:

    results.append(
        doc_store[idx]
    )

print(results)

['to the company’s style and brand guidelines. Nova is an early-stage startup building a suite of generative AI tools designed to protect brand integrity, and today, the company is announcing two new products to help brands police AI-generated content: BrandGuard and BrandGPT. With BrandGuard, you ingest your company’s brand guidelines and style guide, and with a series of models Nova has created, it can check the content against those rules to make sure it’s in compliance, while BrandGPT lets you ask questions about the brand’s content rules in ChatGPT style. Rob May, founder and CEO at the company, who previously founded Backupify, a cloud backup startup that was acquired by Datto back in 2014, recognized that companies wanted to start taking advantage of generative AI technology to create content faster, but they still worried about maintaining brand integrity, so he came up with the idea of building a guard rail system to protect the brand from generative AI mishaps. “We heard from

In [23]:
#building user defined function for whole search flow
def search(query, k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype('float32'),
        k
    )

    results = []

    for idx in indices[0]:

        results.append(doc_store[idx])

    return results

In [24]:
search('Christine paired with whom and wrote about what?',k = 5)

['Kendall Roy enters a conference room with his siblings. As the scene opens, he takes a seat and declares: “Who will be the successor? Me.” Of course, that scene didn’t appear on HBO’s hit show, but it’s a good illustration of generative AI’s level of sophistication compared to the real thing. Yet as the Writers Guild of America goes on strike in pursuit of livable working conditions and better streaming residuals, the networks won’t budge on writers’ demands to regulate the use of AI in writers’ rooms. “Our proposal is that we not be required to adapt something that’s output by AI, and that the output of an AI not be considered writers’ work,” comedy writer Adam Conover told TechCrunch. “That doesn’t entirely exclude that technology from the production process, but it does mean that our working conditions wouldn’t be undermined by AI.” But the Alliance of Motion Picture and Television Producers (AMPTP) refused to engage with that proposal, instead offering a yearly meeting to discuss

In [25]:
query = 'What is news about Google I/O'

In [26]:
#reranking model's output with crossencoder, that compares query with each output and gives score
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

pairs = [
    [query, chunk]
    for chunk in search(query)
]

scores = reranker.predict(pairs)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [27]:
scores

array([0.33286685, 0.6420371 , 0.71982294], dtype=float32)

In [28]:
ranked = sorted(
    zip(scores, chunks),
    reverse=True
)

In [29]:
ranked

[(np.float32(0.71982294),
  'The Interchange! If you received this in your inbox, thank you for signing up and your vote of confidence. If you’re reading this as a post on our site, sign up here so you can receive it directly in the future. Every week, we’ll take a look at the hottest fintech news of the previous week. This will include everything from funding rounds to trends to an analysis of a particular space to hot takes on a particular company or phenomenon. There’s a lot of fintech news out there and it’s our job to stay on top of it — and make sense of it — so you can stay in the know. — Mary Ann and Christine Busy, busy, busy It was a busy week in startup and venture lands, and the fintech space was no exception. In the venture world, I reported on Peter Ackerson’s departure from Fin Capital earlier this year and the fact that he has since started a new venture firm called Audere Capital. The circumstances around his departure remain fuzzy, but one source speculated that tensi

In [11]:
#storing embedding in a numpy file
np.save(
    "/content/drive/MyDrive/myvectordb/embeddings.npy",
    embedding
)

In [31]:
#saving chunks in a file
import json
with open("/content/drive/MyDrive/myvectordb/doc_store.json","w") as f:
  json.dump(doc_store,f)

In [32]:
#save FAISS index
faiss.write_index(
    index,
    "/content/drive/MyDrive/myvectordb/vector.index"
)

We have built a persistent VectorDB, enabling vector search and efficient retrieval with reranking